#  FuseLens Hybrid Pipeline: Long-Read Discovery & Deep Learning Validation

This notebook implements an end-to-end clinical workflow for DNA Breakpoint Detection.

### The Workflow
1.  **Discovery (CTAT-LR-fusion):** We process raw **Nanopore/PacBio FASTQ** files to identify potential fusion isoforms based on structural alignment.
2.  **Context Extraction:** We extract the full 20kb genomic context around the detected breakpoints. This step bridges the gap between noisy long reads and our foundation model trained on clean reference genomes.
3.  **FuseLens Validation:** We load the **pre-trained FuseLens model** to:
    * **Classify:** Confirm if the sequence contains a true fusion motif (Filter out alignment artifacts).
    * **Refine:** Use Attention Pooling to pinpoint the exact breakpoint location.

---

## 1. Environment Setup & Requirements

**System Requirements:**
* **CTAT-LR-fusion:** Must be installed separately (requires `minimap2` and `samtools`). [GitHub Link](https://github.com/TrinityCTAT/CTAT-LR-fusion)
* **Python Libraries:** `pysam` (for genome handling), `transformers`, `torch`, `pandas`.
* **Data:**
    * Reference Genome (`.fa` and `.fai` index)
    * Raw Nanopore FASTQ file
    * Trained FuseLens Model Path

In [ ]:
# Install Python dependencies
!pip install -q pysam torch transformers pandas tqdm safetensors

import os
import pandas as pd
import pysam
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import numpy as np

# ============ CONFIGURATION - UPDATE THESE PATHS ============

# 1. Input Data
NANOPORE_FASTQ = "./data/patient_sample.fastq.gz"  # Your raw long-read data
REF_GENOME = "./data/GRCh38.primary_assembly.genome.fa" # Indexed Reference Genome

# 2. CTAT Configuration (Paths to tool)
CTAT_OUTPUT_DIR = "./ctat_output"

# 3. FuseLens Model (Points to the folder containing model.safetensors from training)
FUSELENS_MODEL_PATH = "./hyenadna_breakpoint_model/final_model"

# 4. Settings
CONTEXT_SIZE = 10000 # 10kb on each side (Total 20kb)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Running on: {DEVICE}")

## 2. Step 1: Run CTAT-LR-fusion (Discovery)

We use `CTAT-LR-fusion` to map the long reads and find structural candidates.

*Note: If you have already run CTAT and have the `fusions.tsv` file, you can skip executing this cell.*

In [ ]:
# Define the command
ctat_cmd = f"""
ctat-LR-fusion \
 --left_fq {NANOPORE_FASTQ} \
 --genome_lib_dir ./ctat_genome_lib_build_dir \
 --output_dir {CTAT_OUTPUT_DIR} \
 --vis
"""

# Execute (Uncomment to run if tool is installed)
# print("🚀 Running CTAT-LR-fusion... This may take time.")
# !{ctat_cmd}

# Check for output
ctat_results_file = os.path.join(CTAT_OUTPUT_DIR, "ctat-LR-fusion.fusion_predictions.tsv")
if not os.path.exists(ctat_results_file):
    print(f"⚠️ Warning: CTAT output not found at {ctat_results_file}.")
    print("   Please ensure CTAT-LR-fusion has run successfully or update the path.")
else:
    print(f"✅ Found CTAT results: {ctat_results_file}")

## 3. Step 2: The "Bridge" - Context Extraction

CTAT gives us coordinates. However, raw Nanopore reads have high error rates (5-10%).
To validate efficiently, we extract the **clean genomic sequence** around these coordinates. This asks FuseLens: *"Does this genomic location look like a valid fusion breakpoint?"*

In [ ]:
def get_reverse_complement(seq):
    trans = str.maketrans("ATCGN", "TAGCN")
    return seq.translate(trans)[::-1]

def extract_fuselens_input(ctat_tsv_path, ref_fasta_path, output_csv="fuselens_input.csv"):
    """
    Parses CTAT output and extracts 20kb genomic context for FuseLens.
    """
    print("📂 Loading CTAT results...")
    try:
        df = pd.read_csv(ctat_tsv_path, sep="\t")
    except FileNotFoundError:
        print("❌ CTAT File not found. Cannot proceed.")
        return None

    print("🧬 Opening Reference Genome...")
    if not os.path.exists(ref_fasta_path):
        print(f"❌ Reference genome not found at {ref_fasta_path}")
        return None
        
    fasta = pysam.FastaFile(ref_fasta_path)
    
    model_inputs = []
    
    print(f"Processing {len(df)} candidates...")
    for idx, row in df.iterrows():
        # CTAT Format: chr1:12345:+
        try:
            l_chr, l_pos, l_strand = row['LeftBreakpoint'].split(':')
            r_chr, r_pos, r_strand = row['RightBreakpoint'].split(':')
            
            # Extract 5' Sequence (Head) - Ends at breakpoint
            # Fetch 10kb UPSTREAM
            seq_head = fasta.fetch(l_chr, int(l_pos) - CONTEXT_SIZE, int(l_pos)).upper()
            if l_strand == '-': seq_head = get_reverse_complement(seq_head)
                
            # Extract 3' Sequence (Tail) - Starts at breakpoint
            # Fetch 10kb DOWNSTREAM
            seq_tail = fasta.fetch(r_chr, int(r_pos), int(r_pos) + CONTEXT_SIZE).upper()
            if r_strand == '-': seq_tail = get_reverse_complement(seq_tail)
            
            # FuseLens Input (20kb)
            full_sequence = seq_head + seq_tail
            
            model_inputs.append({
                'FusionName': row['FusionName'],
                'LeftBreakpoint': row['LeftBreakpoint'],
                'RightBreakpoint': row['RightBreakpoint'],
                'sequence': full_sequence
            })
        except Exception as e:
            print(f"⚠️ Error processing row {idx}: {e}")

    result_df = pd.DataFrame(model_inputs)
    result_df.to_csv(output_csv, sep='\t', index=False)
    print(f"✅ Generated {len(result_df)} FuseLens-ready sequences: {output_csv}")
    return result_df

# Run the extractor
# NOTE: Ensure paths are correct before running
fuselens_input_df = extract_fuselens_input(ctat_results_file, REF_GENOME)

## 4. Step 3: Load FuseLens Model

We reconstruct the model class and load the weights saved from your training pipeline.

In [ ]:
class HyenaDNAClassifier(nn.Module):
    """
    Reconstruction of the trained model architecture.
    """
    def __init__(self, model_name: str, num_labels: int = 2):
        super(HyenaDNAClassifier, self).__init__()
        
        # Use bfloat16 if available, same as training
        torch_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
        
        self.hyenadna = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=torch_dtype
        )
        
        self.hidden_size = self.hyenadna.config.d_model
        self.attention_weights = nn.Linear(self.hidden_size, 1)
        
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels)
        ).to(torch.float32)

    def forward(self, input_ids, attention_mask=None, labels=None, output_attentions=False):
        # Pass ONLY input_ids to backbone
        outputs = self.hyenadna(input_ids) 
        
        if hasattr(outputs, 'last_hidden_state'):
            sequence_output = outputs.last_hidden_state.to(torch.float32)
        else:
            sequence_output = outputs[0].to(torch.float32)

        # Apply mask for attention pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            sequence_output = sequence_output * mask
        
        # Attention Pooling
        attn_scores = self.attention_weights(sequence_output)
        
        if attention_mask is not None:
            attn_mask = (attention_mask == 0).unsqueeze(-1)
            attn_scores = attn_scores.masked_fill(attn_mask, float('-inf'))
        
        attn_probs = torch.softmax(attn_scores, dim=1)
        pooled_output = torch.sum(sequence_output * attn_probs, dim=1)
        
        logits = self.classifier(pooled_output)
        
        return_dict = {'logits': logits}
        
        # Return attention peak index for breakpoint localization
        if output_attentions:
            return_dict['breakpoint_index'] = torch.argmax(attn_probs, dim=1).squeeze(-1)
            
        return return_dict

print("✅ Model class defined.")

## 5. Step 4: Run Inference

We load the trained weights and process the sequences extracted in Step 2.

In [ ]:
def run_fuselens_inference(input_df, model_path, batch_size=8):
    if input_df is None or input_df.empty:
        print("⚠️ No data to process.")
        return

    print(f"🚀 Loading FuseLens from {model_path}...")
    
    # 1. Load Tokenizer & Model
    # Note: We use the base model config for init, then load weights
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = HyenaDNAClassifier("LongSafari/hyenadna-small-32k-seqlen-hf", num_labels=2)
    
    # Load state dict (weights)
    # Try safetensors first, then pytorch_model.bin
    try:
        from safetensors.torch import load_model
        load_model(model, os.path.join(model_path, "model.safetensors"))
    except:
        print("Safetensors not found, trying pytorch_model.bin...")
        model.load_state_dict(torch.load(os.path.join(model_path, "pytorch_model.bin")))
        
    model.to(DEVICE)
    model.eval()

    # 2. Prepare Dataset
    class InferenceDataset(Dataset):
        def __init__(self, sequences):
            self.sequences = sequences
        def __len__(self): return len(self.sequences)
        def __getitem__(self, idx):
            seq = self.sequences[idx]
            return tokenizer(seq, truncation=True, max_length=20480, padding='max_length', return_tensors='pt')

    dataset = InferenceDataset(input_df['sequence'].tolist())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    # 3. Predict
    print("🔮 Validating candidates...")
    probs_list = []
    breakpoints_list = []
    
    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch['input_ids'].squeeze(1).to(DEVICE)
            attention_mask = batch['attention_mask'].squeeze(1).to(DEVICE)
            
            # Run model with attention output
            outputs = model(input_ids, attention_mask=attention_mask, output_attentions=True)
            
            # Probabilities
            probs = torch.softmax(outputs['logits'], dim=-1)[:, 1] # Class 1 (Positive)
            probs_list.extend(probs.cpu().numpy())
            
            # Breakpoint Refinement (Indices)
            breakpoints_list.extend(outputs['breakpoint_index'].cpu().numpy())

    # 4. Attach Results
    input_df['FuseLens_Confidence'] = probs_list
    input_df['Refined_Breakpoint_Index'] = breakpoints_list
    
    # Logic: If Index is near 10,000 (center), it aligns with CTAT. 
    # Calculate offset: (Index - 10000)
    input_df['Breakpoint_Offset_bp'] = input_df['Refined_Breakpoint_Index'] - CONTEXT_SIZE
    
    return input_df

# Run
final_results = run_fuselens_inference(fuselens_input_df, FUSELENS_MODEL_PATH)

## 6. Step 5: Clinical Reporting

Filter the results to show only high-confidence fusions.

In [ ]:
if final_results is not None:
    # Filter: Confidence > 0.90
    high_conf = final_results[final_results['FuseLens_Confidence'] > 0.90].copy()
    
    print("="*60)
    print(f"CLINICAL REPORT: Found {len(high_conf)} High-Confidence Fusions")
    print("="*60)
    
    # Display clean table
    display_cols = ['FusionName', 'LeftBreakpoint', 'FuseLens_Confidence', 'Breakpoint_Offset_bp']
    print(high_conf[display_cols].to_markdown(index=False))
    
    # Save
    high_conf.to_csv("final_clinical_report.csv", index=False)
    print("\n✅ Report saved to 'final_clinical_report.csv'")